# CIDDS-001 — M4 Frozen GSD–PI Validation

This notebook directly downloads the **official CIDDS-001 ZIP** from Hochschule Coburg to **Colab temporary storage**, extracts the **OpenStack** cohort, computes **unique-destination source out-degree**, and applies **unchanged** M3 cutoffs before reading the flow labels.

- It does not download anything to your local PC.
- Frozen policy: `δ_TC = 0.05`, `γ_min = 3`, `k_min = 2`, `top_k = 30`, MGR weak `<2`, boundary `[2,5)`, strong `≥5`.
- `normal` is a confirmed negative source label only when **all** flows from that source are normal; `attacker` ever is a positive; victim/suspicious/unknown source records remain unknown.
- An `ABSTAIN` is **not** detection failure nor success. Structural labels do **not** prove attacker selectivity.
- Official dataset: https://www.hs-coburg.de/forschen/cidds-coburg-intrusion-detection-data-sets/

**Outputs:** `cidds001_m4_frozen_aggregate.csv`, `cidds001_m4_topk_known_labels.csv`, `policy.json`, saved under `MyDrive/paper2_gsd_results/cidds001_m4`. No IP addresses are exported. Only these aggregated results need to be uploaded here.

**Independence note:** OpenStack weeks belong to ONE source capture environment, not independent datasets. CIDDS-001 is distinct from CTU-13; the two together may serve as independent structural stress-test environments, not necessarily decisive evidence of attack-selectivity prediction.

In [5]:
%pip -q install duckdb requests pandas numpy

In [6]:
# CIDDS-001 M4: FROZEN GSD–PI VALIDATION. Intended to run on Google Colab.
from pathlib import Path
import requests, zipfile, shutil, re, json, hashlib, math
import duckdb, numpy as np, pandas as pd

ROOT=Path('/content/cidds001_m4'); ROOT.mkdir(parents=True,exist_ok=True)
URL='https://www.hs-coburg.de/wp-content/uploads/2024/11/CIDDS-001.zip'
ARCHIVE=ROOT/'CIDDS-001.zip'; OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
POLICY=dict(delta_tc=0.05,gamma_min=3.0,k_min=2,top_k=30,mgr_low=2.0,mgr_strong=5.0)
policy_hash=hashlib.sha256(json.dumps(POLICY,sort_keys=True).encode()).hexdigest()
print('FROZEN SETTINGS:',POLICY, 'SHA256:',policy_hash)

# Download directly to temporary Colab storage, never to the user's computer.
if not ARCHIVE.exists() or not zipfile.is_zipfile(ARCHIVE):
    temp=ROOT/'download.part'
    with requests.get(URL,stream=True,timeout=(30,120),headers={'User-Agent':'Mozilla/5.0'}) as r:
        r.raise_for_status()
        with temp.open('wb') as out:
            for chunk in r.iter_content(1024*1024):
                if chunk: out.write(chunk)
    if not zipfile.is_zipfile(temp): raise ValueError('Official URL did not return a ZIP file')
    temp.replace(ARCHIVE)
print('Official ZIP MB:',round(ARCHIVE.stat().st_size/1e6,1))

# Use ONLY OpenStack; ExternalServer has many unknown/suspicious traffic labels.
extracted=[]; target=ROOT/'openstack'; target.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    members=[m for m in z.infolist() if not m.is_dir()]
    for i,m in enumerate(members):
        name=m.filename.replace('\\','/')
        if 'openstack' in name.lower() and name.lower().endswith('.csv'):
            dest=target/f'{i:04d}_{Path(name).name}'
            if not dest.exists() or dest.stat().st_size!=m.file_size:
                with z.open(m) as src,dest.open('wb') as dst: shutil.copyfileobj(src,dst,1024*1024)
            extracted.append(dest)
if not extracted:
    print('Entries:',[m.filename for m in members[:40]])
    raise FileNotFoundError('Could not identify OpenStack CSV files; inspect archive structure first')
print('OpenStack files:',[p.name for p in extracted])

def quote(s): return '"'+s.replace('"','""')+'"'
def lit(s): return "'"+str(s).replace("'","''")+"'"
def colpick(cols,aliases):
    idx={re.sub('[^a-z0-9]','',x.lower()):x for x in cols}
    for key in aliases:
        v=idx.get(re.sub('[^a-z0-9]','',key.lower()))
        if v: return v
    raise ValueError(f'Cannot locate {aliases}; columns = {cols}')
scan='read_csv(['+','.join(lit(p) for p in extracted)+'],header=true,auto_detect=true,all_varchar=true,union_by_name=true,sample_size=2000)'
con=duckdb.connect(str(ROOT/'work.duckdb'))
cols=con.execute('DESCRIBE SELECT * FROM '+scan).df()['column_name'].tolist()
print('CSV columns:',cols)
s=quote(colpick(cols,['Src IP Addr','Src IP','SrcAddr']))
d=quote(colpick(cols,['Dst IP Addr','Dst IP','Dest IP','DstAddr']))
c=quote(colpick(cols,['class','Label']))

# Phase 1: read ONLY source and destination fields; no class labels.
con.execute('DROP VIEW IF EXISTS network')
con.execute(f'''CREATE TEMP VIEW network AS SELECT NULLIF(TRIM({s}),'') src,
            NULLIF(TRIM({d}),'') dst FROM {scan}''')
con.execute('DROP TABLE IF EXISTS degree')
con.execute('''CREATE TABLE degree AS SELECT src,COUNT(DISTINCT dst)::BIGINT out_degree
            FROM network WHERE src IS NOT NULL AND dst IS NOT NULL GROUP BY src''')
ent=con.execute('SELECT src,out_degree FROM degree').df()
x=ent['out_degree'].to_numpy(dtype=np.int64)

def gini(a):
    a=np.sort(a.astype(float));n=len(a)
    return float(2*np.dot(np.arange(1,n+1),a)/(n*a.sum())-(n+1)/n)
q90,q95,q97=np.percentile(x,[90,95,97],method='linear')
tc=bool(abs(q97-q90)/max(q97,1)<POLICY['delta_tc'])
top=np.sort(x[x>1])[::-1][:POLICY['top_k']]
ratios=top[:-1]/top[1:] if len(top)>1 else np.array([])
mgr=float(max(ratios)) if len(ratios) else float('nan')
gaps=[(i+1,float(r),int(top[i])) for i,r in enumerate(ratios) if r>POLICY['gamma_min'] and i+1>=POLICY['k_min']]
if not np.isfinite(mgr): isolation='INSUFFICIENT_TAIL'
elif mgr<2: isolation='WEAK_EXTREME_TAIL'
elif mgr<5: isolation='INCONCLUSIVE_BOUNDARY'
else: isolation='STRONG_EXTREME_TAIL_CANDIDATE'
rank=None;tau=None
if not tc: decision='P95_STABLE';tau=float(q95)
elif mgr<2: decision='ABSTAIN_WEAK_COLLAPSE'
elif mgr<5: decision='ABSTAIN_BOUNDARY'
elif gaps: decision='GAP_RECOVERY_CANDIDATE';rank,_,tau=gaps[0]
else: decision='ABSTAIN_NO_VALID_GAP'
mask=(x>=tau) if tau is not None else np.zeros(len(x),dtype=bool)
print('LABEL-FREE RESULT:',dict(n_sources=len(x),gini=round(gini(x),6),p95=q95,TC=tc,MGR=mgr,decision=decision,isolation=isolation,selected=int(mask.sum()) if tau is not None else 'ABSTAIN'))

# Phase 2: only NOW reveal flow labels. Victim/suspicious/unknown must NOT be called benign.
con.execute('DROP VIEW IF EXISTS flow_labels')
con.execute(f'''CREATE TEMP VIEW flow_labels AS SELECT NULLIF(TRIM({s}),'') src,
            LOWER(TRIM({c})) category FROM {scan}''')
con.execute('DROP TABLE IF EXISTS source_truth')
con.execute('''CREATE TABLE source_truth AS SELECT src,
            MAX(CASE WHEN category='attacker' THEN 1 ELSE 0 END) AS ever_attacker,
            MIN(CASE WHEN category='normal' THEN 1 ELSE 0 END) AS all_normal
            FROM flow_labels WHERE src IS NOT NULL GROUP BY src''')
labels=con.execute('''SELECT d.out_degree,
            CASE WHEN s.ever_attacker=1 THEN 1 WHEN s.all_normal=1 THEN 0 ELSE -1 END AS truth
            FROM degree d LEFT JOIN source_truth s ON d.src=s.src''').df()
y=labels['truth'].fillna(-1).to_numpy(dtype=int)
x2=labels['out_degree'].to_numpy(dtype=int)
selected=(x2>=tau) if tau is not None else np.zeros(len(x2),dtype=bool)
counts={j:int((y==j).sum()) for j in [-1,0,1]}
sel={j:int(((y==j)&selected).sum()) if tau is not None else None for j in [-1,0,1]}
precision=(sel[1]/(sel[0]+sel[1])) if tau is not None and sel[-1]==0 and sel[0]+sel[1]>0 else math.nan
summary=dict(dataset='CIDDS-001',cohort='OpenStack (all CSV weeks)',flow_count=int(con.execute('SELECT COUNT(*) FROM network').fetchone()[0]),
    n_active_sources=len(x),unique_directed_edges=int(x.sum()),Gini_active=gini(x),CV_active=float(x.std()/x.mean()),
    P90=float(q90),P95=float(q95),P97=float(q97),P95_median=float(q95/np.median(x)),
    TC=tc,MGR=mgr,valid_gap_ranks=';'.join(str(a[0]) for a in gaps),chosen_gap_rank=rank,
    selected_threshold=tau,selection_decision=decision,PI_isolation_diagnostic=isolation,
    n_selected=int(selected.sum()) if tau is not None else None,
    TailDensity_P95=float(np.mean(x>=q95)),TailDensity_selected=float(selected.mean()) if tau is not None else None,
    known_attack_sources=counts[1],normal_only_sources=counts[0],unknown_sources=counts[-1],
    attacker_sources_selected=sel[1],normal_sources_selected=sel[0],unknown_sources_selected=sel[-1],
    precision_if_complete_source_labels=precision,policy_sha256=policy_hash)
pd.DataFrame([summary]).to_csv(OUT/'cidds001_m4_frozen_aggregate.csv',index=False)
rows=[]
for k in [5,10,20,50]:
    n=min(k,len(x2));cut=int(np.partition(x2,len(x2)-n)[len(x2)-n])
    above=x2>cut;ties=x2==cut; remaining=n-int(above.sum())
    a=int(((above)&(y==1)).sum())
    rows.append(dict(k=n,cutoff=cut,tie_count=int(ties.sum()),
       known_attacker_min=a+max(0,remaining-int(((ties)&(y!=1)).sum())),
       known_attacker_max=a+min(remaining,int(((ties)&(y==1)).sum())),
       note='Bounds for CONFIRMED attackers, not precision when labels unknown'))
pd.DataFrame(rows).to_csv(OUT/'cidds001_m4_topk_known_labels.csv',index=False)
(OUT/'policy.json').write_text(json.dumps({'policy':POLICY,'sha256':policy_hash,'source':URL,'cohort':'OpenStack only'},indent=2))
print('LABEL AUDIT:',dict(known_attack=counts[1],normal_only=counts[0],unknown=counts[-1]))
print('AGGREGATE FILES:',[p.name for p in OUT.iterdir()])

from google.colab import drive
drive.mount('/content/drive')
target=Path('/content/drive/MyDrive/paper2_gsd_results/cidds001_m4');target.mkdir(parents=True,exist_ok=True)
for f in OUT.iterdir(): shutil.copy2(f,target/f.name)
print('Results saved in Google Drive:',target)


FROZEN SETTINGS: {'delta_tc': 0.05, 'gamma_min': 3.0, 'k_min': 2, 'top_k': 30, 'mgr_low': 2.0, 'mgr_strong': 5.0} SHA256: 6a7ef701ec380f2ead47f03c16d912afb92f489e7663286b631e9ba15a0f2bc1
Official ZIP MB: 402.7
OpenStack files: ['0045_CIDDS-001-internal-week1.csv', '0046_CIDDS-001-internal-week2.csv', '0047_CIDDS-001-internal-week3.csv', '0048_CIDDS-001-internal-week4.csv']
CSV columns: ['Date first seen', 'Duration', 'Proto', 'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Packets', 'Bytes', 'Flows', 'Flags', 'Tos', 'class', 'attackType', 'attackID', 'attackDescription']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LABEL-FREE RESULT: {'n_sources': 17020, 'gini': 0.770779, 'p95': np.float64(19.0), 'TC': False, 'MGR': 66.0, 'decision': 'P95_STABLE', 'isolation': 'STRONG_EXTREME_TAIL_CANDIDATE', 'selected': 1172}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LABEL AUDIT: {'known_attack': 2, 'normal_only': 16998, 'unknown': 20}
AGGREGATE FILES: ['policy.json', 'cidds001_m4_topk_known_labels.csv', 'cidds001_m4_frozen_aggregate.csv']
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results saved in Google Drive: /content/drive/MyDrive/paper2_gsd_results/cidds001_m4


In [7]:
!pip -q install "duckdb>=1.0,<2" pandas numpy requests

In [9]:
"""CIDDS-001 OpenStack weekly temporal stability under the pre-existing FROZEN GSD–PI M3 policy.

Runs in Google Colab. No source or destination IPs are exported.
The algorithm ignores labels until all structural outputs have been finalized.
The four weeks are temporal partitions of ONE dataset, not independent test environments.
SQL syntax fix: use week_id in DuckDB; week is a reserved keyword. Frozen policy unchanged.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import sys
import zipfile

import numpy as np
import pandas as pd
import requests
import duckdb

ROOT = Path(os.environ.get('CIDDS_ROOT', '/content/cidds001_m4'))
OPENSTACK = ROOT / 'openstack'
RESULTS = ROOT / 'temporal_results'
ARCHIVE = ROOT / 'CIDDS-001.zip'
URL = 'https://www.hs-coburg.de/wp-content/uploads/2024/11/CIDDS-001.zip'
for p in (ROOT,OPENSTACK,RESULTS): p.mkdir(parents=True,exist_ok=True)
POLICY = dict(delta_tc=0.05,gamma_min=3.0,k_min=2,top_k=30,mgr_low=2.0,mgr_strong=5.0)
POLICY_HASH = hashlib.sha256(json.dumps(POLICY,sort_keys=True).encode()).hexdigest()
EXPECTED_HASH = '6a7ef701ec380f2ead47f03c16d912afb92f489e7663286b631e9ba15a0f2bc1'
assert POLICY_HASH == EXPECTED_HASH, 'Policy was modified! This is not a frozen experiment.'
print('FROZEN POLICY (unchanged):',POLICY)
print('Policy SHA-256:',POLICY_HASH)


def week_no(name: str):
    m=re.search(r'week[\s_\-]*([1-4])(?=\D|$)',str(name).lower())
    return int(m.group(1)) if m else None


def locate_week_files():
    # Reuse the OpenStack CSVs extracted by the prior CIDDS-001 Colab notebook.
    paths=sorted(p for p in OPENSTACK.glob('*.csv') if week_no(p.name))
    existing={w:[] for w in range(1,5)}
    for p in paths: existing[week_no(p.name)].append(p)
    if all(existing.values()):
        print('Reusing already extracted OpenStack weekly CSVs; no download needed.')
        return existing
    if not ARCHIVE.exists() or not zipfile.is_zipfile(ARCHIVE):
        print('Local ZIP not available. Downloading official CIDDS-001 ZIP directly in Colab...')
        temp=ROOT/'download.part'
        with requests.get(URL,stream=True,timeout=(30,120),headers={'User-Agent':'Mozilla/5.0'}) as r:
            r.raise_for_status()
            with temp.open('wb') as out:
                for b in r.iter_content(1024*1024):
                    if b: out.write(b)
        if not zipfile.is_zipfile(temp):
            raise RuntimeError('Downloaded file was not a ZIP archive')
        temp.replace(ARCHIVE)
    with zipfile.ZipFile(ARCHIVE) as z:
        for i,member in enumerate(z.infolist()):
            path=member.filename.replace('\\','/')
            if member.is_dir() or not path.lower().endswith('.csv') or 'openstack' not in path.lower():
                continue
            w=week_no(path)
            if not w: continue
            dest=OPENSTACK/f'{i:04d}_{Path(path).name}'
            if not dest.exists() or dest.stat().st_size!=member.file_size:
                print('Extracting:',Path(path).name)
                with z.open(member) as src,dest.open('wb') as dst: shutil.copyfileobj(src,dst,1024*1024)
    files={w:[] for w in range(1,5)}
    for p in sorted(OPENSTACK.glob('*.csv')):
        w=week_no(p.name)
        if w: files[w].append(p)
    if not all(files.values()):
        raise FileNotFoundError(f'Missing weeks: {[w for w,fs in files.items() if not fs]}; openstack filenames={[p.name for p in OPENSTACK.glob("*.csv")]}')
    return files


def sql_lit(s): return "'"+str(s).replace("'","''")+"'"
def sql_ident(s): return '"'+str(s).replace('"','""')+'"'


def pick(columns,aliases):
    normalized={re.sub(r'[^a-z0-9]','',x.lower()):x for x in columns}
    for alias in aliases:
        chosen=normalized.get(re.sub(r'[^a-z0-9]','',alias.lower()))
        if chosen is not None: return sql_ident(chosen)
    raise ValueError(f'Columns {columns} do not contain any of {aliases}')


def gini(a):
    b=np.sort(a.astype(np.float64));n=len(b)
    return float((2*np.dot(np.arange(1,n+1),b)/(n*b.sum()))-(n+1)/n)


def run_protocol(x,cohort,flows):
    assert len(x)>0 and (x>0).all(), (cohort,len(x))
    p90,p95,p97=np.percentile(x,[90,95,97],method='linear')
    tc=bool(abs(p97-p90)/max(p97,1)<POLICY['delta_tc'])
    top=np.sort(x[x>1])[::-1][:POLICY['top_k']]
    ratios=top[:-1]/top[1:] if len(top)>1 else np.array([],dtype=np.float64)
    mgr=float(np.max(ratios)) if len(ratios) else math.nan
    eligible=[(i+1,float(r),int(top[i])) for i,r in enumerate(ratios)
              if r>POLICY['gamma_min'] and i+1>=POLICY['k_min']]
    if not np.isfinite(mgr): iso='INSUFFICIENT_TAIL'
    elif mgr<POLICY['mgr_low']: iso='WEAK_EXTREME_TAIL'
    elif mgr<POLICY['mgr_strong']: iso='INCONCLUSIVE_BOUNDARY'
    else: iso='STRONG_EXTREME_TAIL_CANDIDATE'
    threshold=None; rank=None
    if not tc:
        decision='P95_STABLE';threshold=float(p95)
    elif not np.isfinite(mgr): decision='ABSTAIN_INSUFFICIENT_TAIL'
    elif mgr<POLICY['mgr_low']: decision='ABSTAIN_WEAK_COLLAPSE'
    elif mgr<POLICY['mgr_strong']: decision='ABSTAIN_BOUNDARY'
    elif eligible:
        decision='GAP_RECOVERY_CANDIDATE';rank,_,threshold=eligible[0]
    else: decision='ABSTAIN_NO_VALID_GAP'
    selected=(x>=threshold) if threshold is not None else np.zeros(len(x),dtype=bool)
    return dict(cohort=cohort,n_flows_with_valid_pair=int(flows),n_active_sources=int(len(x)),
        n_distinct_edges=int(x.sum()),Gini_active=gini(x),CV_active=float(x.std()/x.mean()),
        P90=float(p90),P95=float(p95),P97=float(p97),P95_over_median=float(p95/np.median(x)),
        TC=tc,MGR=float(mgr),valid_gap_ranks=';'.join(str(g[0]) for g in eligible),
        chosen_gap_rank=rank,selected_threshold=threshold,selection_decision=decision,
        PI_isolation_diagnostic=iso,n_selected=int(selected.sum()) if threshold is not None else None,
        TailDensity_P95=float(np.mean(x>=p95)),
        TailDensity_selected=float(selected.mean()) if threshold is not None else None,
        policy_sha256=POLICY_HASH)


def topk_bounds(degrees,labels,k,cohort):
    n=len(degrees);k=min(k,n);boundary=int(np.partition(degrees,n-k)[n-k]);above=degrees>boundary; ties=degrees==boundary
    strict_n=int(above.sum());remaining=k-strict_n;tie_n=int(ties.sum())
    row=dict(cohort=cohort,k=k,cutoff_degree=boundary,strictly_above=strict_n,
             degree_ties_at_cutoff=tie_n,tie_crosses_boundary=bool(tie_n>remaining))
    for v,name in [(1,'attacker'),(0,'normal_only'),(-1,'unknown')]:
        n_above=int(((labels==v)&above).sum());n_tie=int(((labels==v)&ties).sum())
        row[f'{name}_min']=n_above+max(0,remaining-(tie_n-n_tie))
        row[f'{name}_max']=n_above+min(remaining,n_tie)
    return row


def main():
    week_files=locate_week_files()
    for w,fs in week_files.items():
        if len(fs) != 1:
            raise RuntimeError(f'Week {w} has {len(fs)} OpenStack CSVs; inspect for duplicate extraction before continuing: {[p.name for p in fs]}')
        print('Week',w,':', [(p.name,round(p.stat().st_size/1e6,1)) for p in fs])
    # Use previous notebook's CSV parsing settings so cohort and methodology match.
    one=week_files[1][0]
    sample=f'read_csv({sql_lit(one)},header=true,auto_detect=true,all_varchar=true,union_by_name=true,sample_size=2000)'
    con=duckdb.connect(str(ROOT/'weekly_stability.duckdb'))
    con.execute("SET memory_limit='5GB'")
    cols=con.execute('DESCRIBE SELECT * FROM '+sample).df()['column_name'].tolist()
    source_col=pick(cols,['Src IP Addr','Src IP','SrcAddr'])
    dest_col=pick(cols,['Dst IP Addr','Dst IP','Dest IP','DstAddr'])
    class_col=pick(cols,['class','Label'])
    print('Source / destination / label columns matched:', source_col, dest_col, class_col)
    subqueries=[]
    for week,fs in sorted(week_files.items()):
        for f in fs:
            table=f'read_csv({sql_lit(f)},header=true,auto_detect=true,all_varchar=true,union_by_name=true,sample_size=2000)'
            subqueries.append((week,table))
    union_struct=' UNION ALL '.join(f'''SELECT {w}::SMALLINT AS week_id,
                NULLIF(TRIM({source_col}),'') src,NULLIF(TRIM({dest_col}),'') dst
                FROM {t}''' for w,t in subqueries)
    # PHASE ONE -- NO LABEL COLUMNS ARE READ.
    con.execute('DROP TABLE IF EXISTS weekly_edges')
    print('PHASE 1: building source→distinct-destination edges without accessing labels...')
    con.execute(f'''CREATE TABLE weekly_edges AS
        SELECT week_id,src,dst,COUNT(*)::BIGINT AS n_flows FROM ({union_struct}) input
        WHERE src IS NOT NULL AND dst IS NOT NULL GROUP BY week_id,src,dst''')
    con.execute('DROP TABLE IF EXISTS source_degrees')
    con.execute('''CREATE TABLE source_degrees AS
        SELECT week_id,src,COUNT(*)::BIGINT AS out_degree FROM weekly_edges GROUP BY week_id,src
        UNION ALL SELECT 0::SMALLINT AS week_id,src,COUNT(DISTINCT dst)::BIGINT AS out_degree
        FROM weekly_edges GROUP BY src''')
    results=[]
    for week in [0,1,2,3,4]:
        label='Pooled' if week==0 else f'Week {week}'
        degrees=con.execute('SELECT out_degree FROM source_degrees WHERE week_id=?',[week]).df()['out_degree'].to_numpy(dtype=np.int64)
        flows=int(con.execute('SELECT COALESCE(SUM(n_flows),0) FROM weekly_edges '+('' if week==0 else 'WHERE week_id=?'), ([] if week==0 else [week])).fetchone()[0])
        data=run_protocol(degrees,label,flows)
        data['week_id']=week
        results.append(data)
        print('LABEL-FREE',label,':',dict(sources=data['n_active_sources'],edges=data['n_distinct_edges'],Gini=round(data['Gini_active'],5),P95=data['P95'],MGR=data['MGR'],TC=data['TC'],decision=data['selection_decision'],selected=data['n_selected']))
    structural=pd.DataFrame(results)
    structural.to_csv(RESULTS/'cidds001_m4_weekly_structure_label_free.csv',index=False)
    print('Saved LABEL-FREE decisions before evaluating any label data.')
    pooled=results[0]
    original=dict(flows=31287933,sources=17020,edges=201778,Gini=0.7707785949769068, P95=19.0, MGR=66.0,selected=1172)
    compare=dict(
        flows_match=pooled['n_flows_with_valid_pair']==original['flows'],
        sources_match=pooled['n_active_sources']==original['sources'],
        edges_match=pooled['n_distinct_edges']==original['edges'],
        gini_match=abs(pooled['Gini_active']-original['Gini'])<1e-10,
        P95_match=pooled['P95']==original['P95'],
        MGR_match=pooled['MGR']==original['MGR'],
        selection_match=pooled['n_selected']==original['selected'])
    if not all(compare.values()):
        print('WARNING: Pooled dataset differs from prior run; investigate prior to claims:',compare)
    else: print('Pooled reproduction verified against prior independent-run aggregate.')
    # PHASE TWO -- decode labels only AFTER structure CSV and decision log are fixed.
    union_labels=' UNION ALL '.join(f'''SELECT {w}::SMALLINT AS week_id,
        NULLIF(TRIM({source_col}),'') src,LOWER(TRIM({class_col})) category
        FROM {t}''' for w,t in subqueries)
    con.execute('DROP TABLE IF EXISTS weekly_source_truth')
    print('PHASE 2: reading labels for POST-HOC source-level auditing...')
    con.execute(f'''CREATE TABLE weekly_source_truth AS SELECT week_id,src,
          MAX(CASE WHEN category='attacker' THEN 1 ELSE 0 END)::INTEGER ever_attacker,
          MIN(CASE WHEN category='normal' THEN 1 ELSE 0 END)::INTEGER all_normal
          FROM ({union_labels}) input WHERE src IS NOT NULL GROUP BY week_id,src''')
    con.execute('DROP TABLE IF EXISTS source_truth')
    con.execute('''CREATE TABLE source_truth AS SELECT * FROM weekly_source_truth
        UNION ALL SELECT 0::SMALLINT AS week_id,src,MAX(ever_attacker)::INTEGER ever_attacker,
        MIN(all_normal)::INTEGER all_normal FROM weekly_source_truth GROUP BY src''')
    # CHECK: label reading cannot change any of the structural decisions above.
    extra=[];tk=[]
    for week in [0,1,2,3,4]:
        label='Pooled' if week==0 else f'Week {week}'
        ent=con.execute('''SELECT d.out_degree,
            CASE WHEN t.ever_attacker=1 THEN 1
                 WHEN t.all_normal=1 THEN 0 ELSE -1 END::INTEGER truth
            FROM source_degrees d LEFT JOIN source_truth t
              ON d.week_id=t.week_id AND d.src=t.src
            WHERE d.week_id=?''',[week]).df()
        x=ent['out_degree'].to_numpy(dtype=np.int64)
        y=ent['truth'].fillna(-1).to_numpy(dtype=np.int64)
        row=results[week].copy();tau=row['selected_threshold']
        selected=(x>=tau) if tau is not None else None
        row.update(known_attack_sources=int((y==1).sum()),normal_only_sources=int((y==0).sum()),unknown_sources=int((y==-1).sum()))
        if selected is not None:
            at=int((selected&(y==1)).sum());normal=int((selected&(y==0)).sum());unknown=int((selected&(y==-1)).sum());n=int(selected.sum())
            row.update(attacker_sources_selected=at,normal_sources_selected=normal,unknown_sources_selected=unknown,
                precision_lower_bound=at/n if n else math.nan,
                precision_upper_bound=(at+unknown)/n if n else math.nan,
                # NOT attack recall: source truth has mixed classes and unknowns.
                known_attack_source_capture=at/int((y==1).sum()) if (y==1).any() else math.nan)
        else:
            row.update(attacker_sources_selected=None,normal_sources_selected=None,unknown_sources_selected=None,
                precision_lower_bound=math.nan,precision_upper_bound=math.nan,known_attack_source_capture=math.nan)
        extra.append(row)
        for k in [5,10,20,50]:
            tk.append(topk_bounds(x,y,k,label))
        print('POST-HOC',label, 'known_attack=',row['known_attack_sources'], 'normal_only=',row['normal_only_sources'], 'unknown=',row['unknown_sources'],
              'selected_attacker=',row['attacker_sources_selected'])
    pd.DataFrame(extra).to_csv(RESULTS/'cidds001_m4_weekly_aggregate.csv',index=False)
    pd.DataFrame(tk).to_csv(RESULTS/'cidds001_m4_weekly_topk_tie_bounds.csv',index=False)
    audit=dict(dataset='CIDDS-001',capture='OpenStack',units='active source IP; out-degree = number of distinct destination IPs',
        timeline='4 files partitioned by original OpenStack week (plus pooled control)',
        frozen_policy=POLICY,policy_sha256=POLICY_HASH,
        first_stage='Label-free GSD–PI decisions computed and saved before reading any label column',
        second_stage='Attack if any outgoing flow class=attacker; normal only if every outgoing flow class=normal; otherwise unknown',
        note='Four weeks are dependent temporal partitions, NOT four independently collected datasets',
        abstain='Undefined precision/recall: no alert set emitted',
        known_attack_zero='When no attack sources are present, attack recall is undefined',
        topk='Tie-aware descriptive ranges, NOT an output of Algorithm 1',
        pooled_reproduction=compare)
    (RESULTS/'cidds001_m4_temporal_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
    print('\nFILES:',[f.name for f in sorted(RESULTS.iterdir())])
    if os.environ.get('CIDDS_SKIP_DRIVE') != '1':
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            target=Path('/content/drive/MyDrive/paper2_gsd_results/cidds001_m4/temporal')
            target.mkdir(parents=True,exist_ok=True)
            for f in RESULTS.iterdir():
                if f.is_file():shutil.copy2(f,target/f.name)
            print('Exported aggregate files to Google Drive:',target)
        except ImportError:
            print('Google Colab not detected, files remain in',RESULTS)
    con.close()
    return structural, pd.DataFrame(extra)


if __name__=='__main__':
    main()


FROZEN POLICY (unchanged): {'delta_tc': 0.05, 'gamma_min': 3.0, 'k_min': 2, 'top_k': 30, 'mgr_low': 2.0, 'mgr_strong': 5.0}
Policy SHA-256: 6a7ef701ec380f2ead47f03c16d912afb92f489e7663286b631e9ba15a0f2bc1
Reusing already extracted OpenStack weekly CSVs; no download needed.
Week 1 : [('0045_CIDDS-001-internal-week1.csv', 1167.2)]
Week 2 : [('0046_CIDDS-001-internal-week2.csv', 1435.2)]
Week 3 : [('0047_CIDDS-001-internal-week3.csv', 840.7)]
Week 4 : [('0048_CIDDS-001-internal-week4.csv', 817.9)]
Source / destination / label columns matched: "Src IP Addr" "Dst IP Addr" "class"
PHASE 1: building source→distinct-destination edges without accessing labels...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LABEL-FREE Pooled : {'sources': 17020, 'edges': 201778, 'Gini': 0.77078, 'P95': 19.0, 'MGR': 66.0, 'TC': False, 'decision': 'P95_STABLE', 'selected': 1172}
LABEL-FREE Week 1 : {'sources': 9346, 'edges': 98731, 'Gini': 0.76446, 'P95': 17.0, 'MGR': 28.513513513513512, 'TC': False, 'decision': 'P95_STABLE', 'selected': 588}
LABEL-FREE Week 2 : {'sources': 9333, 'edges': 101158, 'Gini': 0.76386, 'P95': 18.0, 'MGR': 38.05714285714286, 'TC': False, 'decision': 'P95_STABLE', 'selected': 515}
LABEL-FREE Week 3 : {'sources': 8599, 'edges': 92718, 'Gini': 0.76142, 'P95': 18.0, 'MGR': 29.57894736842105, 'TC': False, 'decision': 'P95_STABLE', 'selected': 447}
LABEL-FREE Week 4 : {'sources': 8317, 'edges': 88670, 'Gini': 0.75916, 'P95': 17.0, 'MGR': 27.342105263157894, 'TC': False, 'decision': 'P95_STABLE', 'selected': 565}
Saved LABEL-FREE decisions before evaluating any label data.
Pooled reproduction verified against prior independent-run aggregate.
PHASE 2: reading labels for POST-HOC source-le

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

POST-HOC Pooled known_attack= 2 normal_only= 16998 unknown= 20 selected_attacker= 2
POST-HOC Week 1 known_attack= 1 normal_only= 9325 unknown= 20 selected_attacker= 1
POST-HOC Week 2 known_attack= 1 normal_only= 9312 unknown= 20 selected_attacker= 1
POST-HOC Week 3 known_attack= 0 normal_only= 8599 unknown= 0 selected_attacker= 0
POST-HOC Week 4 known_attack= 0 normal_only= 8317 unknown= 0 selected_attacker= 0

FILES: ['cidds001_m4_temporal_audit.json', 'cidds001_m4_weekly_aggregate.csv', 'cidds001_m4_weekly_structure_label_free.csv', 'cidds001_m4_weekly_topk_tie_bounds.csv']
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Exported aggregate files to Google Drive: /content/drive/MyDrive/paper2_gsd_results/cidds001_m4/temporal
